# IndoML T2 — E10-R (RunPod A100): big MP-SENet fine-tune (40k mixtures, full MUSAN, bf16, up to 3 s segments, warm start from E9-A) + ensemble submission on the pod
Before running: paste the HF token and the Kaggle API token (KGAT_...) in cell 0. Run cells top to bottom. Download the model and the ZIP **before** stopping the pod.

In [ ]:
# 0) Setup on RunPod — paste your tokens below (do not share this notebook afterwards)
import os, sys, glob, json, subprocess, stat
os.environ['HF_TOKEN'] = 'PASTE_YOUR_HF_TOKEN_HERE'
os.environ['KAGGLE_API_TOKEN'] = 'PASTE_YOUR_KAGGLE_API_TOKEN_HERE'      # the KGAT_... token from Kaggle Settings -> API Tokens
assert os.environ['HF_TOKEN'].startswith('hf_'), 'paste your Hugging Face token'
assert os.environ['KAGGLE_API_TOKEN'].startswith('KGAT_'), 'paste your Kaggle API token'
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
tok = os.path.expanduser('~/.kaggle/access_token'); open(tok, 'w').write(os.environ['KAGGLE_API_TOKEN']); os.chmod(tok, stat.S_IRUSR | stat.S_IWUSR)
for p_ in ['/workspace/work', '/workspace/input', '/kaggle']: os.makedirs(p_, exist_ok=True)
for src, dst in [('/workspace/work', '/kaggle/working'), ('/workspace/input', '/kaggle/input')]:
    if not os.path.exists(dst): os.symlink(src, dst)          # reuse the Kaggle code paths unchanged
subprocess.run('pip -q install -U kaggle jiwer sentencepiece webrtcvad-wheels librosa einops soundfile pandas scipy', shell=True)
print(subprocess.run('kaggle --version', shell=True, capture_output=True, text=True).stdout)
import torch; print(torch.__version__, torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1e9), 'GB')

In [ ]:
# 1) Pull our private Kaggle datasets with the Kaggle API (no manual uploads)
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
mine = {d.ref.split('/')[1]: d.ref for d in api.dataset_list(mine=True, page=1)}
print('my datasets:', sorted(mine))
def get(slug, dest):
    os.makedirs(dest, exist_ok=True); print('downloading', mine[slug], '->', dest, flush=True)
    api.dataset_download_files(mine[slug], path=dest, unzip=True, quiet=True)
get('indoml-validation', '/workspace/input/datasets/avin55/indoml-validation')
get('indoml-track2-test', '/workspace/input/datasets/avin55/indoml-track2-test')
get('musan-noise', '/workspace/input/musan-noise')
get('indoml-t2-ckpt', '/workspace/input/indoml-t2-ckpt')
# E9 checkpoints from the E9 notebook output (only the .pt files + held_clips.json)
os.makedirs('/workspace/input/e9', exist_ok=True)
for pat in ['e9_dns_init.pt', 'e9_warm_e8.pt', 'held_clips.json']:
    subprocess.run(f'kaggle kernels output avics55/notebookb50abb10c9 -p /workspace/input/e9 --file-pattern "{pat}"', shell=True)
chk = {'validation metadata': glob.glob('/workspace/input/**/validationMetadata.json', recursive=True),
       'test wavs (3178)': len(glob.glob('/workspace/input/datasets/avin55/indoml-track2-test/audio/*.wav')),
       'musan wavs (930)': len(glob.glob('/workspace/input/musan-noise/**/*.wav', recursive=True)),
       'noisy_asr_val.json': glob.glob('/workspace/input/**/noisy_asr_val.json', recursive=True),
       'e9_dns_init.pt': glob.glob('/workspace/input/e9/**/e9_dns_init.pt', recursive=True),
       'e9_warm_e8.pt': glob.glob('/workspace/input/e9/**/e9_warm_e8.pt', recursive=True)}
for k, v in chk.items(): print(f'{k:22s}:', v if v else 'MISSING')
print('If an e9 file is MISSING: download it from the E9 notebook Output page and upload it into /workspace/input/e9/ via the Jupyter file panel.')

In [ ]:
!rm -rf /kaggle/working/ner && git clone -q --depth 1 https://github.com/nagshubhadip/noise-event-detection-and-removal /kaggle/working/ner
!pip -q install jiwer sentencepiece webrtcvad-wheels librosa 2>&1 | tail -1
!wget -q https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip -O /kaggle/working/esc50.zip && unzip -q -o /kaggle/working/esc50.zip -d /kaggle/working/ && ls /kaggle/working/ESC-50-master | head
# ---- MP-SENet (MIT) ----
!rm -rf /kaggle/working/mpsenet && git clone -q --depth 1 https://github.com/yxlu-0102/MP-SENet /kaggle/working/mpsenet
!pip -q install einops 2>&1 | tail -1

In [ ]:
import sys, os, glob, json, time, random, numpy as np, torch, soundfile as sf, librosa
from pathlib import Path
VAL = Path('/kaggle/input/datasets/avin55/indoml-validation/validation')
M = Path('/kaggle/working/ner/track2_removal/versions/v-1_indoml2026_track2/modular')
d = (M/'data.py').read_text()
d = d.replace('Path("/kaggle/input/indoml-validation"),', 'Path("/kaggle/input/datasets/avin55/indoml-validation/validation"), Path("/kaggle/input/indoml-validation"),')
d = d.replace('cid = (r.get("clip_id") or r.get("id")', 'cid = (r.get("segmentFileName") or r.get("clip_id") or r.get("id")')
(M/'data.py').write_text(d); sys.path.insert(0, str(M))
HF = os.environ["HF_TOKEN"]
import config, data, evaluate
import model as model_mod
from train import train_on_sim
from inference import enhance, Transcriber
from jiwer import wer as jwer
config.ensure_dirs()
val_dir, items = data.resolve_validation()
meta = {m['segmentFileName'][:-4]: m for m in json.load(open(VAL/'validationMetadata.json', encoding='utf-8'))}
print('validation clips', len(items))
import torch.nn.functional as F

In [ ]:
import types, sys
sys.modules.setdefault('pesq', types.SimpleNamespace(pesq=lambda *a, **k: 0.0))   # PESQ not needed (no metric discriminator)
sys.path.insert(0, '/kaggle/working/mpsenet')
from env import AttrDict
from dataset import mag_pha_stft, mag_pha_istft
from models.model import MPNet, phase_losses
h = AttrDict(json.load(open('/kaggle/working/mpsenet/best_ckpt/config.json')))
dev = torch.device('cuda')
def load_mp(path='/kaggle/working/mpsenet/best_ckpt/g_best_dns'):
    m = MPNet(h).to(dev); m.load_state_dict(torch.load(path, map_location=dev, weights_only=False)['generator']); return m.eval()
def mp_forward(m, wav):                       # wav [B, T] (RMS-normalised) -> enhanced [B, T]
    amp, pha, _ = mag_pha_stft(wav, h.n_fft, h.hop_size, h.win_size, h.compress_factor)
    amp_g, pha_g, com_g = m(amp, pha)
    return mag_pha_istft(amp_g, pha_g, h.n_fft, h.hop_size, h.win_size, h.compress_factor), (amp_g, pha_g, com_g)
@torch.no_grad()
def enhance_mp(m, x, win=4.0, hop=3.0):
    x = np.asarray(x, np.float32); n = len(x); sr = 16000
    nf = float(np.sqrt(n / (np.sum(x**2) + 1e-9))); xn = torch.from_numpy(x * nf).to(dev)
    W, H = int(win * sr), int(hop * sr)
    if n <= W:
        y = mp_forward(m, xn[None])[0][0][:n]
    else:
        y = torch.zeros(n, device=dev); wsum = torch.zeros(n, device=dev)
        fade = torch.ones(W, device=dev); r = W - H; fade[:r] = torch.linspace(0, 1, r, device=dev); fade[-r:] = torch.linspace(1, 0, r, device=dev)
        s = 0
        while True:
            e = min(s + W, n); seg = xn[s:e]
            if len(seg) < W: seg = torch.nn.functional.pad(seg, (0, W - len(seg)))
            out = mp_forward(m, seg[None])[0][0][:W]
            L = e - s; f = fade[:L].clone()
            if s == 0: f[:r] = 1
            if e == n: f[-min(r, L):] = 1
            y[s:e] += out[:L] * f; wsum[s:e] += f
            if e == n: break
            s += H
        y = y / wsum.clamp_min(1e-6)
    y = (y / nf).cpu().numpy()
    return np.pad(y, (0, max(0, n - len(y))))[:n].astype(np.float32)
mp = load_mp(); print('MP-SENet params (M):', round(sum(p.numel() for p in mp.parameters()) / 1e6, 2))

In [ ]:
# Isolated validation noise events + fingerprint clustering
SR = 16000
NV = Path('/kaggle/working/bank/val_noise'); NV.mkdir(parents=True, exist_ok=True)
segs, fps, owner = [], [], []
for v in items:
    if not v['syn']: continue
    y = data.read_audio(v['path']); c = data.read_audio(v['ref']); L = min(len(y), len(c)); n = (y[:L] - c[:L]).astype(np.float32)
    for k, e in enumerate(meta.get(v['id'], {}).get('NoiseSubCategoryTimeStamp', [])):
        a, b = max(0, int((float(e['start']) - 0.02) * SR)), min(L, int((float(e['end']) + 0.02) * SR))
        if b - a < int(0.3 * SR) or np.abs(n[a:b]).max() < 1e-4: continue
        seg = n[a:b]; p = NV / f"{v['id'][-40:]}_{k}.wav"; sf.write(p, seg, SR)
        mel = librosa.power_to_db(librosa.feature.melspectrogram(y=seg, sr=SR, n_mels=48, n_fft=512, hop_length=160) + 1e-10)
        f = librosa.util.fix_length(mel, size=48, axis=1).ravel(); f = (f - f.mean()) / (f.std() + 1e-6)
        segs.append({'path': str(p), 'dur': (b - a) / SR}); fps.append(f / np.linalg.norm(f)); owner.append(v['id'])
F = np.stack(fps); cents, lab = [], np.zeros(len(F), int)
for i, f in enumerate(F):
    if cents:
        s = np.stack(cents) @ f; j = int(s.argmax())
        if s[j] > 0.95: lab[i] = j; continue
    cents.append(f); lab[i] = len(cents) - 1
ncl = len(cents); rng = np.random.default_rng(7)
print('noise events', len(F), '| clusters', ncl)
for frac in [0.35, 0.45, 0.55]:
    held = set(rng.permutation(ncl)[:int(frac * ncl)].tolist())
    clip_cl = {}
    for o, l in zip(owner, lab): clip_cl.setdefault(o, set()).add(int(l))
    HELD_CLIPS = [o for o, s in clip_cl.items() if s <= held]
    if len(HELD_CLIPS) >= 80: break
train_noise = [s for s, l in zip(segs, lab) if int(l) not in held]
print(f'held-out clusters {len(held)}/{ncl} | held-out eval clips {len(HELD_CLIPS)} | train noise segments {len(train_noise)}')

In [ ]:
# ESC-50 noise segments (active part, 0.3-2.5 s), resampled to 16 kHz
E = Path('/kaggle/working/ESC-50-master'); NE = Path('/kaggle/working/bank/esc50'); NE.mkdir(parents=True, exist_ok=True)
import pandas as pd
esc = pd.read_csv(E/'meta/esc50.csv'); esc_noise = []
for _, r in esc.iterrows():
    w, _ = librosa.load(E/'audio'/r.filename, sr=SR, mono=True)
    w, _ = librosa.effects.trim(w, top_db=30)
    if len(w) < int(0.3 * SR): continue
    w = w[:int(2.5 * SR)].astype(np.float32); p = NE / r.filename; sf.write(p, w, SR)
    esc_noise.append({'path': str(p), 'dur': len(w) / SR, 'cat': r.category})
print('ESC-50 segments', len(esc_noise), '| categories', esc.category.nunique())

In [ ]:
# MUSAN noise from the uploaded Kaggle dataset (no download)
MUF = sorted(glob.glob('/kaggle/input/**/free-sound/*.wav', recursive=True) + glob.glob('/kaggle/input/**/sound-bible/*.wav', recursive=True))
print('MUSAN wav files found:', len(MUF), '(expect 930)'); assert len(MUF) > 800, 'attach the musan-noise dataset'
NM = Path('/kaggle/working/bank/musan'); NM.mkdir(parents=True, exist_ok=True); rngm = np.random.default_rng(3); musan_noise = []
for k, fpath in enumerate(MUF):
    w, _ = librosa.load(fpath, sr=SR, mono=True)
    if len(w) < int(0.5 * SR) or np.abs(w).max() < 1e-4: continue
    for j in range(min(4, max(1, len(w) // (3 * SR)))):
        L = int(rngm.uniform(0.5, 2.5) * SR); st = int(rngm.integers(0, max(1, len(w) - L)))
        seg = w[st:st + L].astype(np.float32)
        if np.sqrt((seg**2).mean()) < 1e-3: continue
        p = NM / f'{k:04d}_{j}.wav'; sf.write(p, seg, SR); musan_noise.append({'path': str(p), 'dur': len(seg) / SR})
print('MUSAN noise segments:', len(musan_noise))

In [ ]:
# Noise augmentation: speed perturbation, random EQ, synthetic reverb -> one augmented copy per base segment
from scipy.signal import butter, sosfilt, fftconvolve
NA = Path('/kaggle/working/bank/aug'); NA.mkdir(parents=True, exist_ok=True); rnga = np.random.default_rng(11)
def augment(w):
    f = rnga.choice([0.9, 1.0, 1.1]);  w = librosa.resample(w, orig_sr=int(SR * f), target_sr=SR) if f != 1.0 else w
    if rnga.random() < 0.6:
        if rnga.random() < 0.5: sos = butter(4, rnga.uniform(2000, 7000), 'low', fs=SR, output='sos')
        else: sos = butter(2, rnga.uniform(100, 500), 'high', fs=SR, output='sos')
        w = sosfilt(sos, w)
    if rnga.random() < 0.5:
        rt = rnga.uniform(0.15, 0.7); t = np.arange(int(rt * SR)) / SR
        rir = rnga.standard_normal(len(t)) * np.exp(-6.9 * t / rt); rir[0] = 1.0
        w = fftconvolve(w, rir / np.abs(rir).sum() * 4)[:len(w)]
    pk = np.abs(w).max(); return (w / pk * 0.9 if pk > 0 else w).astype(np.float32)
base = train_noise + esc_noise + musan_noise; aug_noise = []
for k, nb_ in enumerate(base):
    w, _ = sf.read(nb_['path'], dtype='float32'); w = augment(w)
    if len(w) < int(0.3 * SR): continue
    p = NA / f'a{k:05d}.wav'; sf.write(p, w, SR); aug_noise.append({'path': str(p), 'dur': len(w) / SR})
print('noise pool:', len(base), 'base +', len(aug_noise), 'augmented')

In [ ]:
SCRIPT = r'''
import sys, os, json, time, glob, argparse, types, numpy as np, torch, torch.nn.functional as F
ap = argparse.ArgumentParser(); ap.add_argument("--lam", type=float); ap.add_argument("--epochs", type=int, default=5)
ap.add_argument("--out"); ap.add_argument("--bs", type=int, default=2); ap.add_argument("--seg", type=int, default=16000)
ap.add_argument("--lr", type=float, default=2e-4); ap.add_argument("--hours", type=float, default=9.0)
ap.add_argument("--init", default="/kaggle/working/mpsenet/best_ckpt/g_best_dns"); ap.add_argument("--bf16", type=int, default=0); a = ap.parse_args()
M = "/kaggle/working/ner/track2_removal/versions/v-1_indoml2026_track2/modular"; sys.path.insert(0, M)
sys.modules.setdefault("pesq", types.SimpleNamespace(pesq=lambda *x, **k: 0.0)); sys.path.append("/kaggle/working/mpsenet")   # append: MP-SENet also has an inference.py
HF = os.environ["HF_TOKEN"]
import config, data, evaluate
import model as model_mod
from inference import Transcriber
from jiwer import wer as jwer
from env import AttrDict
from dataset import mag_pha_stft, mag_pha_istft
from models.model import MPNet, phase_losses
torch.manual_seed(0); np.random.seed(0)
h = AttrDict(json.load(open("/kaggle/working/mpsenet/best_ckpt/config.json"))); dev = torch.device("cuda")
def load_mp(path="/kaggle/working/mpsenet/best_ckpt/g_best_dns"):
    m = MPNet(h).to(dev); m.load_state_dict(torch.load(path, map_location=dev, weights_only=False)["generator"]); return m
def mp_forward(m, wav):
    amp, pha, _ = mag_pha_stft(wav, h.n_fft, h.hop_size, h.win_size, h.compress_factor)
    a, p, c = m(amp, pha); return mag_pha_istft(a, p, h.n_fft, h.hop_size, h.win_size, h.compress_factor), (a, p, c)
@torch.no_grad()
def enhance_mp(m, x, win=4.0, hop=3.0):
    x = np.asarray(x, np.float32); n = len(x); sr = 16000
    nf = float(np.sqrt(n / (np.sum(x**2) + 1e-9))); xn = torch.from_numpy(x * nf).to(dev); W, H = int(win*sr), int(hop*sr)
    if n <= W: y = mp_forward(m, xn[None])[0][0][:n]
    else:
        y = torch.zeros(n, device=dev); ws = torch.zeros(n, device=dev); r = W - H
        fade = torch.ones(W, device=dev); fade[:r] = torch.linspace(0, 1, r, device=dev); fade[-r:] = torch.linspace(1, 0, r, device=dev); s = 0
        while True:
            e = min(s + W, n); seg = xn[s:e]
            if len(seg) < W: seg = F.pad(seg, (0, W - len(seg)))
            out = mp_forward(m, seg[None])[0][0][:W]; L = e - s; f = fade[:L].clone()
            if s == 0: f[:r] = 1
            if e == n: f[-min(r, L):] = 1
            y[s:e] += out[:L] * f; ws[s:e] += f
            if e == n: break
            s += H
        y = y / ws.clamp_min(1e-6)
    y = (y / nf).cpu().numpy(); return np.pad(y, (0, max(0, n - len(y))))[:n].astype(np.float32)
def post(x, y):
    ri, ro = np.sqrt((x**2).mean()), np.sqrt((y**2).mean()); z = y * (ri / ro) if ro > 1e-8 else y
    pk = np.abs(z).max(); return (z * (0.99 / pk) if pk > 0.99 else z).astype(np.float32)
records = [json.loads(l) for l in open(config.MANIFEST)]
tdl, _, _, _ = data.build_dataloaders(records, a.bs)
encoder = enc_fwd = None
if a.lam > 0: _, encoder, _ = model_mod.core_model(HF); enc_fwd = type(encoder).forward.__wrapped__
val_dir, items = data.resolve_validation(); byid = {v["id"]: v for v in items}
H = json.load(open("/kaggle/working/held_clips.json")); X = {i: data.read_audio(byid[i]["path"]) for i in H}; REF = {i: data.read_audio(byid[i]["ref"]) for i in H}
def heldout_sisdr(m):
    m.eval(); s = float(np.mean([evaluate.si_sdr(REF[i], post(X[i], enhance_mp(m, X[i]))) for i in H])); m.train(); return s
m = load_mp(a.init); m.train(); opt = torch.optim.AdamW(m.parameters(), lr=a.lr, betas=(0.8, 0.99))
total = a.epochs * len(tdl); sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=a.lr, total_steps=total, pct_start=0.05)
best = heldout_sisdr(m); print(f"[lam={a.lam}] init ({os.path.basename(a.init)}) held-out SI-SDR {best:.2f}", flush=True); torch.save({"generator": m.state_dict()}, a.out)
t0 = time.time(); step = 0; stop = False
for ep in range(a.epochs):
    run = None
    for b in tdl:
        mix, clean = b["mix"].to(dev), b["clean"].to(dev)
        if mix.shape[-1] > a.seg: st = np.random.randint(0, mix.shape[-1] - a.seg); mix, clean = mix[:, st:st+a.seg], clean[:, st:st+a.seg]
        nf = torch.sqrt(mix.shape[-1] / (mix.pow(2).sum(-1, keepdim=True) + 1e-9)); mix, clean = mix * nf, clean * nf
        c_amp, c_pha, c_com = mag_pha_stft(clean, h.n_fft, h.hop_size, h.win_size, h.compress_factor)
        n_amp, n_pha, _ = mag_pha_stft(mix, h.n_fft, h.hop_size, h.win_size, h.compress_factor)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=bool(a.bf16)):
            a_g, p_g, co_g = m(n_amp, n_pha)
        a_g, p_g, co_g = a_g.float(), p_g.float(), co_g.float()
        y = mag_pha_istft(a_g, p_g, h.n_fft, h.hop_size, h.win_size, h.compress_factor)[:, :clean.shape[-1]]
        _, _, co_h = mag_pha_stft(y, h.n_fft, h.hop_size, h.win_size, h.compress_factor)
        ip, gd, iaf = phase_losses(c_pha, p_g)
        loss = 0.9*F.mse_loss(c_amp, a_g) + 0.3*(ip+gd+iaf) + 0.2*F.mse_loss(c_com, co_g) + 0.2*F.mse_loss(co_g, co_h) + 0.2*F.l1_loss(clean, y)
        if a.lam > 0:
            with torch.no_grad(): fc = enc_fwd(encoder, clean / nf); fc = F.layer_norm(fc.float(), fc.shape[-1:])
            fe = enc_fwd(encoder, y / nf); fe = F.layer_norm(fe.float(), fe.shape[-1:]); loss = loss + a.lam * (fe - fc).abs().mean()
        opt.zero_grad(set_to_none=True); loss.backward(); gn = torch.nn.utils.clip_grad_norm_(m.parameters(), 5.0)
        if torch.isfinite(gn): opt.step()
        sched.step(); step += 1; run = loss.item() if run is None else 0.99 * run + 0.01 * loss.item()
        if step % 500 == 0: print(f"[lam={a.lam}] ep{ep} step{step}/{total} loss {run:.4f} | {(time.time()-t0)/3600:.2f} h", flush=True)
        if time.time() - t0 > a.hours * 3600: stop = True; break
    s = heldout_sisdr(m); print(f"[lam={a.lam}] epoch {ep} held-out SI-SDR {s:.2f} (best {best:.2f})", flush=True)
    if s > best: best = s; torch.save({"generator": m.state_dict()}, a.out)
    if stop: print("time limit reached", flush=True); break
# final leak-free eval of the best checkpoint: held-out synthetic + ALL natural
del encoder; torch.cuda.empty_cache()
m = load_mp(a.out).eval(); noisy = json.load(open(os.environ["NOISY_ASR"])); asr = Transcriber(HF)
NAT = [v["id"] for v in items if not v["syn"]]; GT = {i: evaluate.normalize(byid[i]["gt"]) for i in H + NAT}
for i in NAT: X[i] = data.read_audio(byid[i]["path"])
def pooled(ids, hyp): ids = [i for i in ids if GT[i]]; return jwer([GT[i] for i in ids], [evaluate.normalize(hyp.get(i, "")) or "@" for i in ids])
ENH = {i: enhance_mp(m, X[i]) for i in H + NAT}
for bl in [1.0]:
    out = {i: post(X[i], bl * ENH[i] + (1 - bl) * X[i]) for i in H + NAT}
    si = float(np.mean([evaluate.si_sdr(REF[i], out[i]) for i in H])); hyp = {}; ids = H + NAT
    for k in range(0, len(ids), 16):
        for i, t in zip(ids[k:k+16], asr.transcribe_many([out[i] for i in ids[k:k+16]])): hyp[i] = t
    ds, dn = 100 * (pooled(H, noisy) - pooled(H, hyp)), 100 * (pooled(NAT, noisy) - pooled(NAT, hyp))
    print("[RESULT]", json.dumps(dict(run=os.path.basename(a.out), lam=a.lam, blend=bl, si_sdr_heldout=round(si, 2), dWER_syn_heldout=round(ds, 2), dWER_nat_all=round(dn, 2), combined_est=round(si + 0.32*ds + 0.68*dn, 2))), flush=True)
'''

In [ ]:
import gc
clean_idx, _ = data.build_banks(11000, hf_token=HF)
noise_pool = train_noise + esc_noise + musan_noise + aug_noise
config.SIM.update(snr_db=(-20.0, 5.0), events=(1, 4), max_noise_sec=2.5, min_noise_sec=0.3)
for f in list(config.MIX_DIR.glob('*.wav')) + list(config.REF_DIR.glob('*.wav')): f.unlink()
records = data.simulate_dataset(clean_idx, noise_pool, 40000)
json.dump(HELD_CLIPS, open('/kaggle/working/held_clips.json', 'w'))
open('/kaggle/working/mp_ft.py', 'w').write(SCRIPT)
del mp; gc.collect(); torch.cuda.empty_cache()
# Find the E8 checkpoint: as a plain file, or as a folder that Kaggle auto-extracted from a dataset upload
import zipfile
A_CK = [x for x in glob.glob('/kaggle/input/**/e9_dns_init.pt', recursive=True) if os.path.isfile(x)]
assert A_CK, 'attach the E9 notebook output (e9_dns_init.pt)'
sd = torch.load(A_CK[0], map_location='cpu', weights_only=False)['generator']; print('warm start from E9-A:', A_CK[0], '| tensors:', len(sd))
print('mixtures', len(records), '| noise pool', len(noise_pool))

In [ ]:
# 2) Find the largest batch / segment that fits in GPU memory (bf16 autocast)
import gc, torch.nn.functional as F
mp_probe = load_mp(A_CK[0]); mp_probe.train(); CFG = None
for bs, seg in [(8, 48000), (6, 48000), (4, 48000), (4, 32000), (2, 32000)]:
    try:
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        mix = torch.randn(bs, seg, device=dev) * 0.1
        amp, pha, _ = mag_pha_stft(mix, h.n_fft, h.hop_size, h.win_size, h.compress_factor)
        with torch.autocast('cuda', dtype=torch.bfloat16):
            a_g, p_g, c_g = mp_probe(amp, pha)
        (a_g.float().mean() + p_g.float().mean() + c_g.float().mean()).backward(); mp_probe.zero_grad(set_to_none=True)
        peak = torch.cuda.max_memory_allocated() / 1e9; print(f'bs {bs} seg {seg/16000:.1f}s -> peak {peak:.1f} GB', flush=True)
        if peak < 0.8 * torch.cuda.get_device_properties(0).total_memory / 1e9: CFG = (bs, seg); break
    except torch.cuda.OutOfMemoryError:
        print(f'bs {bs} seg {seg/16000:.1f}s -> OOM', flush=True)
    finally:
        gc.collect(); torch.cuda.empty_cache()
del mp_probe; gc.collect(); torch.cuda.empty_cache(); print('chosen config:', CFG); assert CFG

In [ ]:
# 3) Train: warm start from E9-A, 40k mixtures, bf16, chosen batch/segment, up to 8 epochs (5.5 h cap). Model saved after every improving epoch.
NOISY = glob.glob('/kaggle/input/**/noisy_asr_val.json', recursive=True)[0]
env = dict(os.environ, CUDA_VISIBLE_DEVICES='0', NOISY_ASR=NOISY, PYTORCH_ALLOC_CONF='expandable_segments:True')
cmd = ['python', '/kaggle/working/mp_ft.py', '--lam', '0', '--out', '/workspace/work/e10r_warmA.pt', '--init', A_CK[0], '--bf16', '1',
       '--bs', str(CFG[0]), '--seg', str(CFG[1]), '--epochs', '8', '--lr', '1e-4', '--hours', '5.5']
print(' '.join(cmd)); p = subprocess.Popen(cmd, stdout=open('/workspace/work/e10r.log', 'w'), stderr=subprocess.STDOUT, env=env)
import time
while p.poll() is None:
    time.sleep(600); L = [l for l in open('/workspace/work/e10r.log').read().splitlines() if l.startswith('[lam')]
    print(time.strftime('%H:%M'), L[-1] if L else '', flush=True)
print('exit', p.returncode); print('\n'.join(l for l in open('/workspace/work/e10r.log').read().splitlines() if ('held-out' in l) or l.startswith(('[RESULT]', 'Traceback')) or 'Error' in l))
print('>>> Download /workspace/work/e10r_warmA.pt to your laptop now (Jupyter file panel) <<<')

In [ ]:
# 4) Pick the best of single models and ensembles on held-out (+400 natural), then build the test submission on this GPU
def find(name):
    c = [x for x in glob.glob(f'/kaggle/input/**/{name}', recursive=True) + glob.glob(f'/workspace/work/{name}') if os.path.isfile(x)]; assert c, name; return c[0]
mA, mB, mC = load_mp(find('e9_dns_init.pt')), load_mp(find('e9_warm_e8.pt')), load_mp('/workspace/work/e10r_warmA.pt')
from inference import check_output, Transcriber
noisy = json.load(open(NOISY)); asr = Transcriber(HF)
byid = {v['id']: v for v in items}; NAT = [v['id'] for v in items if not v['syn']]
rng = np.random.RandomState(0); NS = [NAT[i] for i in rng.choice(len(NAT), 400, replace=False)]
H = [i for i in HELD_CLIPS if i in byid]; ids = H + NS
X = {i: data.read_audio(byid[i]['path']) for i in ids}; REF = {i: data.read_audio(byid[i]['ref']) for i in H}; GT = {i: evaluate.normalize(byid[i]['gt']) for i in ids}
def pooled(s, hyp): s = [i for i in s if GT[i]]; return jwer([GT[i] for i in s], [evaluate.normalize(hyp.get(i, '')) or '@' for i in s])
def post(x, y):
    ri, ro = np.sqrt((x**2).mean()), np.sqrt((y**2).mean()); z = y * (ri / ro) if ro > 1e-8 else y
    pk = np.abs(z).max(); return (z * (0.99 / pk) if pk > 0.99 else z).astype(np.float32)
E = {n: {i: enhance_mp(m, X[i]) for i in ids} for n, m in [('A', mA), ('B', mB), ('C', mC)]}
combos = {'C': ['C'], 'A+C': ['A', 'C'], 'A+B+C': ['A', 'B', 'C'], 'A+B': ['A', 'B']}
RES = {}
for name, parts in combos.items():
    out = {i: post(X[i], sum(E[p][i] for p in parts) / len(parts)) for i in ids}
    si = float(np.mean([evaluate.si_sdr(REF[i], out[i]) for i in H])); hyp = {}
    for k in range(0, len(ids), 32):
        for i, t in zip(ids[k:k+32], asr.transcribe_many([out[i] for i in ids[k:k+32]])): hyp[i] = t
    ds, dn = 100 * (pooled(H, noisy) - pooled(H, hyp)), 100 * (pooled(NS, noisy) - pooled(NS, hyp)); RES[name] = si + 0.32 * ds + 0.68 * dn
    print(dict(combo=name, si_sdr=round(si, 2), dWER_syn=round(ds, 2), dWER_nat400=round(dn, 2), combined_est=round(RES[name], 2)), flush=True)
BEST = max(RES, key=RES.get); MODELS = {'A': mA, 'B': mB, 'C': mC}; print('chosen:', BEST, '(A+B reference LB 14.53)')

In [ ]:
import soundfile as sf, zipfile, time
from collections import Counter
from pathlib import Path
test_files = sorted(Path('/kaggle/input/datasets/avin55/indoml-track2-test/audio').glob('*.wav')); print('test clips', len(test_files))
sub = Path('/workspace/work/t2_submission'); sub.mkdir(exist_ok=True); fails = Counter(); t0 = time.time(); parts = combos[BEST]
for k, p in enumerate(test_files):
    w = data.read_audio(p); y = post(w, sum(enhance_mp(MODELS[q], w) for q in parts) / len(parts)); c = check_output(w, y)
    for key in ['finite', 'len_match', 'mono', 'not_silent', 'level_ok', 'no_clip']:
        if not c[key]: fails[key] += 1
    assert c['finite'] and c['len_match'] and c['mono'], p.name
    sf.write(sub / p.name, y, 16000, subtype='PCM_16')
    if k % 500 == 0: print(k, f'{(time.time()-t0)/60:.1f} min', flush=True)
print('enhanced in', round((time.time()-t0)/60, 1), 'min | fails', dict(fails), '| combo', BEST)
names = [p.name for p in test_files]; texts = {}
for i in range(0, len(names), 32):
    for n, t in zip(names[i:i+32], asr.transcribe_paths([str(sub / n) for n in names[i:i+32]])): texts[n[:-4]] = (t or '').strip()
with open('/workspace/work/transcripts.jsonl', 'w', encoding='utf-8') as f:
    for n in names: f.write(json.dumps({'clip_id': n[:-4], 'text': texts.get(n[:-4], '')}, ensure_ascii=False) + '\n')
zp = '/workspace/work/submission_track2.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('/workspace/work/transcripts.jsonl', 'transcripts.jsonl')
    for n in names: z.write(sub / n, n)
nl = zipfile.ZipFile(zp).namelist(); print('ZIP entries', len(nl), '| wavs', sum(x.endswith('.wav') for x in nl), '| empty transcripts', sum(1 for v in texts.values() if not v), '| MB', round(os.path.getsize(zp)/1e6))
print('>>> Download /workspace/work/submission_track2.zip and e10r_warmA.pt, THEN stop/terminate the pod <<<')